# Vehicle Assembly Anomaly Benchmark

Compares LightGBM, LSTM, and Transformer. Each dataset uses vehicle-level splitting, cross-validation for hyperparameter selection on training data, then one final held-out test evaluation.
    "metadata": {"id": "07a43b37", "language": "markdown"},

In [ ]:
import sys, subprocess, random
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split, StratifiedKFold, ParameterGrid
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
try:
    import lightgbm as lgb
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'lightgbm'])
    import lightgbm as lgb
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Notebook Python:', sys.executable)
print('LightGBM:', lgb.__version__, '| PyTorch:', torch.__version__, '| Device:', DEVICE)

In [ ]:
DATA_DIR = Path.cwd()
DATASETS = {name: DATA_DIR / filename for name, filename in {'base': 'dataset.csv', 'sparse': 'dataset_variant_sparse.csv', 'drift': 'dataset_variant_drift.csv', 'propagation': 'dataset_variant_propagation.csv', 'noisy': 'dataset_variant_noisy.csv'}.items()}
TARGET = 'anomaly_flag'
VEHICLE_ID = 'vehicle_id'
TEST_SIZE = 0.2
N_SPLITS = 3
NUMERIC_COLS = ['cycle_time_sec', 'torque_nm', 'temperature_c', 'vibration_rms', 'pressure_bar', 'force_n', 'position_error_mm', 'voltage_v', 'current_a', 'flow_rate_lpm', 'queue_time_sec', 'ambient_temperature_c', 'humidity_pct']
CATEGORICAL_COLS = ['vehicle_model', 'vehicle_variant', 'station_id', 'shift', 'production_batch']
FEATURE_COLS = NUMERIC_COLS + CATEGORICAL_COLS
for name, path in DATASETS.items():
    print(name, path.exists(), path)

In [ ]:
def load_dataset(path):
    df = pd.read_csv(path).copy()
    for col in NUMERIC_COLS:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    return df

def split_vehicles(df):
    labels = df.groupby(VEHICLE_ID)[TARGET].max().astype(int)
    train_ids, test_ids = train_test_split(labels.index, test_size=TEST_SIZE, random_state=SEED, stratify=labels)
    return df[df[VEHICLE_ID].isin(train_ids)].copy(), df[df[VEHICLE_ID].isin(test_ids)].copy()

def preprocess(train_df, other_df):
    prep = ColumnTransformer([('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scale', StandardScaler())]), NUMERIC_COLS), ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), CATEGORICAL_COLS)])
    return prep.fit_transform(train_df[FEATURE_COLS]).astype('float32'), prep.transform(other_df[FEATURE_COLS]).astype('float32')

def sequences(df, matrix):
    ordered = df.sort_values([VEHICLE_ID, 'station_id'])
    positions = pd.Series(np.arange(len(df)), index=df.index)
    ordered_matrix = matrix[positions.loc[ordered.index].to_numpy()]
    groups = ordered.groupby(VEHICLE_ID, sort=False)
    X = np.stack([ordered_matrix[positions.loc[group.index].to_numpy()] for _, group in groups])
    y = groups[TARGET].max().astype('float32').to_numpy()
    return X, y

def scores(y, p):
    return {'accuracy': accuracy_score(y, p >= .5), 'precision': precision_score(y, p >= .5, zero_division=0), 'recall': recall_score(y, p >= .5, zero_division=0), 'f1': f1_score(y, p >= .5, zero_division=0), 'roc_auc': roc_auc_score(y, p)}

class SequenceModel(nn.Module):
    def __init__(self, inputs, hidden, layers, dropout, kind):
        super().__init__()
        self.kind = kind
        if kind == 'lstm':
            self.encoder = nn.LSTM(inputs, hidden, layers, batch_first=True, dropout=dropout if layers > 1 else 0)
        else:
            self.projection = nn.Linear(inputs, hidden)
            layer = nn.TransformerEncoderLayer(hidden, 2, hidden * 2, dropout, batch_first=True)
            self.encoder = nn.TransformerEncoder(layer, layers)
        self.head = nn.Linear(hidden, 1)
    def forward(self, x):
        z = self.encoder(x)[0] if self.kind == 'lstm' else self.encoder(self.projection(x))
        return self.head(z[:, -1]).squeeze(1)

def fit_torch(X, y, X_eval, params, kind):
    model = SequenceModel(X.shape[2], params['hidden'], params['layers'], params['dropout'], kind).to(DEVICE)
    pos, neg = max(float(y.sum()), 1), max(float(len(y) - y.sum()), 1)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([neg / pos], device=DEVICE))
    optimizer = torch.optim.Adam(model.parameters(), lr=params['lr'])
    loader = DataLoader(TensorDataset(torch.tensor(X), torch.tensor(y)), batch_size=params['batch'], shuffle=True)
    for _ in range(params['epochs']):
        model.train()
        for xb, yb in loader:
            optimizer.zero_grad()
            loss_fn(model(xb.to(DEVICE)), yb.to(DEVICE)).backward()
            optimizer.step()
    model.eval()
    with torch.no_grad():
        return torch.sigmoid(model(torch.tensor(X_eval).to(DEVICE))).cpu().numpy()

GRIDS = {'lightgbm': {'num_leaves': [15, 31], 'learning_rate': [.03, .08], 'n_estimators': [150, 300]}, 'lstm': {'hidden': [32, 64], 'layers': [1, 2], 'dropout': [.1], 'lr': [.001], 'batch': [64], 'epochs': [8]}, 'transformer': {'hidden': [32, 64], 'layers': [1, 2], 'dropout': [.1], 'lr': [.001], 'batch': [64], 'epochs': [8]}}
MODEL_NAMES = ['lightgbm', 'lstm', 'transformer']

In [ ]:
def benchmark(name, df):
    train_df, test_df = split_vehicles(df)
    train_matrix, test_matrix = preprocess(train_df, test_df)
    X, y = sequences(train_df, train_matrix)
    Xt, yt = sequences(test_df, test_matrix)
    cv = StratifiedKFold(N_SPLITS, shuffle=True, random_state=SEED)
    output = []
    for model_name in MODEL_NAMES:
        best_auc, best = -np.inf, None
        for params in ParameterGrid(GRIDS[model_name]):
            fold_auc = []
            for a, b in cv.split(X, y):
                if model_name == 'lightgbm':
                    model = lgb.LGBMClassifier(objective='binary', class_weight='balanced', verbosity=-1, random_state=SEED, **params)
                    model.fit(X[a].reshape(len(a), -1), y[a])
                    p = model.predict_proba(X[b].reshape(len(b), -1))[:, 1]
                else:
                    p = fit_torch(X[a], y[a], X[b], params, model_name)
                fold_auc.append(roc_auc_score(y[b], p))
            if np.mean(fold_auc) > best_auc:
                best_auc, best = float(np.mean(fold_auc)), params
        if model_name == 'lightgbm':
            final = lgb.LGBMClassifier(objective='binary', class_weight='balanced', verbosity=-1, random_state=SEED, **best)
            final.fit(X.reshape(len(X), -1), y)
            p = final.predict_proba(Xt.reshape(len(Xt), -1))[:, 1]
        else:
            p = fit_torch(X, y, Xt, best, model_name)
        result = {'dataset': name, 'model': model_name, 'cv_auc': best_auc, 'best_params': best, **scores(yt, p)}
        output.append(result)
        print(name, model_name, 'CV AUC=', round(best_auc, 4), 'TEST AUC=', round(result['roc_auc'], 4), 'F1=', round(result['f1'], 4), 'Best:', best)
    return output

results = []
for name, path in DATASETS.items():
    results.extend(benchmark(name, load_dataset(path)))
results_df = pd.DataFrame(results)

In [ ]:
display(results_df.sort_values(['dataset', 'roc_auc'], ascending=[True, False]))
summary = results_df.groupby('model')[['cv_auc', 'roc_auc', 'f1', 'recall']].mean().sort_values('roc_auc', ascending=False)
print('Overall ranking by mean held-out test ROC AUC:')
display(summary)
results_df.to_csv(DATA_DIR / 'model_comparison_results.csv', index=False)